# Data Cleaning :- Telco Customer Churn
This notebook uses the Telco-Customer-Churn datset from kaggle.com. source- https://www.kaggle.com/datasets/blastchar/telco-customer-churn 

This notebook focuses on cleaning and preparing the Telco Customer Churn dataset for further analysis.

The main objectives are to:

- Load and inspect the raw dataset
- Check for missing values and data inconsistencies
- Identify and handle duplicate records
- Convert columns to appropriate data types
- Standardize column values where necessary
- Prepare the cleaned dataset for Exploratory Data Analysis (EDA) and SQL analysis

The goal is to ensure that the dataset is accurate, consistent, and ready for reliable analysis.


## 1. SETTING UP THE DATASET
##### Imported libraries like NumPy, Pandas, Os and Pathlib and loaded the dataset 
- The data set have 7043 rows  and 21 columns


In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

file_path = Path("..") / "Data" / "Telco-Customer-Churn.csv"
df = pd.read_csv(file_path)
display(df.head())
print(f"Shape: {df.shape}")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Shape: (7043, 21)


## 2. Initial Data Quality Assessment

Before applying any transformations, the dataset is examined to understand its structure and identify potential data quality issues.

The following aspects are reviewed:

- Data types
- Missing values
- Number of unique values


In [2]:
quality_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Unique Values": df.nunique()
})

display(quality_summary)

,Data Type,Missing Values,Unique Values
customerID,str,0,7043
gender,str,0,2
SeniorCitizen,int64,0,2
Partner,str,0,2
Dependents,str,0,2
tenure,int64,0,73
PhoneService,str,0,2
MultipleLines,str,0,3
InternetService,str,0,3
OnlineSecurity,str,0,3


**Conclusion**: The Dataset do not contain any missing values 

## 3. Missing and Duplicate Value Assessment

The dataset is checked for explicit missing values and duplicate records.

Records with missing values and duplicate rows are removed where necessary to maintain data consistency and prevent duplicate observations from affecting subsequent analysis.



In [3]:
print(f"Total null values: {df.isnull().sum().sum()}")

print(f"\nDuplicate rows: {df.duplicated().sum()}")

Total null values: 0

Duplicate rows: 0


In [4]:
df = df.dropna().drop_duplicates().reset_index(drop=True)
print(f"\nShape after cleaning: {df.shape}")


Shape after cleaning: (7043, 21)


**Conclusion**: There are 0 nulls rows and 0 duplicate columns. Hence the shape of the dataset do not change

## 4. Handling Blank Values in `TotalCharges`

Although no explicit missing values are identified, the `TotalCharges` column contains blank string values. These values need to be identified and handled before the column can be used as a numerical variable.

In [5]:
blank_totalcharges = (df["TotalCharges"].astype(str).str.strip() == "")
blank_records = df.loc[
    blank_totalcharges,
    [
        "customerID",
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "Contract",
        "Churn"
    ]
]
print("Blank TotalCharges records:", blank_records.shape[0])
display(blank_records)


Blank TotalCharges records: 11


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


## 5. Data Type Conversion

The `TotalCharges` column is converted from a string to a numeric data type. Any values that cannot be converted are treated as missing values and handled appropriately.

In [6]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce").fillna(0).reset_index(drop=True)

**conclusion**: There are a total of 11 blank values found in the `TotalCharges` which are replaced by 0 

## 6. Validation of Numerical Values

Key numerical columns are checked for non-positive values to identify potentially invalid records.

Records containing invalid zero or negative values are removed to ensure that the dataset contains meaningful observations for further analysis.


In [7]:
df = df[df["tenure"] > 0]
print(f"Shape: {df.shape}")
df = df[df["MonthlyCharges"] > 0]
print(f"\nShape: {df.shape}")
df = df[df["TotalCharges"] > 0]
print(f"\nShape: {df.shape}")

Shape: (7032, 21)

Shape: (7032, 21)

Shape: (7032, 21)


## 7. Standardizing Categorical Values

The `SeniorCitizen` column is converted from numeric indicators to descriptive categorical values to improve readability and interpretability during the analysis.


In [8]:
df['SeniorCitizen'] = np.where(df['SeniorCitizen'] == 1, 'Yes', 'No')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,No,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,No,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,No,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 8. Saving the cleaned file

In [11]:
df.to_csv(Path("..") / "Data" / "Telco-Customer-Churn-Cleaned.csv", index=False)

## 9. Final Data Validation

The cleaned dataset is reviewed again to confirm that the identified data quality issues have been addressed successfully.

The final validation includes checking:

- Data types
- Missing values
- Duplicate records
- Dataset structure

Since there where no null vallues neither any duplicate values we will use the same dataset for futher analysis